# Multilingual LVC dataset — Italian (v3: pro-drop and preverbal adverbs)


Short-context generator for Italian. 

Note that Italian is a pro drop language so this includes the option to use **null subjects** in the primary active dataset. 
The person/number of the omitted subject are kept in CSV metadata (`person_key`, `subject_number`). You can also generate a matched overt-pronoun comparison by changing one setting.

**Generators:**
1. Active with **null (unexpressed) subjects** by default; six CSVs (three tenses × with/without adverbs).
2. Passive six CSVs.
3. Active with **nominal subjects**, with configurable determiner. Default 10-sentence preview sample.

**Tenses:** *futuro semplice* (`future`), *passato prossimo* (`present_perfect`) and *trapassato prossimo* (`past_perfect`).

### Run instructions
Place the notebook alongside these input files and run all cells:

- `collocation_items_IT_merged_controls_v4_gender_number_utf8.csv`
- `italian_verbs_inflection.csv`
- `sentence_buildersIT_fixed_utf8.csv`

Python standard library only. Default outputs go into `IT_generated_sentences_v2/` in Excel-compatible UTF-8 CSV format.

**Core settings:** `REALIZE_SUBJECT_PRONOUNS = False` (default); `ACTIVE_PRONOUN_KEYS = ('loro',)` (third-person plural); `MAX_TIME_SPECS = 6`; five adverbs. To generate the same items with explicit pronouns, set `REALIZE_SUBJECT_PRONOUNS = True` and change `OUTPUT_DIR` to a separate folder. The nominal-subject generator is unchanged and still produces a 10-row sample by default.

**Source-data limitations:** This is template-based generation. Repeated input items are sentence-deduplicated; the incomplete expression `ricevere un abbraccio da` is excluded. Possessive *avere* and *fare carriera* are excluded from passives. The passive data need manual linguistic validation.

## 0. Configuration and helper functions

Run this cell first. The default active condition omits the Italian subject pronoun. Change `REALIZE_SUBJECT_PRONOUNS` to `True` for an overt-pronoun comparison.


In [ ]:
"""Italian LVC short-context sentence generator — v3 (preverbal adverbs, null subjects).

Adapted from the French v4 / German v1 experimental design.
Uses Python's standard library, preserves source metadata, and writes UTF-8-SIG CSVs.

Inputs (place next to the notebook/script):
  collocation_items_IT_merged_controls_v4_gender_number_utf8.csv
  italian_verbs_inflection.csv
  sentence_buildersIT_fixed_utf8.csv
"""

from __future__ import annotations

import csv
import random
import re
from collections import Counter
from pathlib import Path

# ===== USER SETTINGS =====
INPUT_DIR = Path('.')
OUTPUT_DIR = Path('IT_generated_sentences_v3')
ITEMS_FILE = 'collocation_items_IT_merged_controls_v4_gender_number_utf8.csv'
VERBS_FILE = 'italian_verbs_inflection.csv'
BUILDER_FILE = 'sentence_buildersIT_fixed_utf8.csv'

TENSES = ('future', 'present_perfect', 'past_perfect')
TENSE_COL = {
    'future': 'future_active',
    'present_perfect': 'present_perfect_active',
    'past_perfect': 'past_perfect_active',
}
PASSIVE_TENSE_COL = {
    'future': 'future_passive',
    'present_perfect': 'present_perfect_passive',
    'past_perfect': 'past_perfect_passive',
}
ACTIVE_PRONOUN_KEYS = ('loro',)  # Default person key (3rd person plural), regardless of subject realization.
REALIZE_SUBJECT_PRONOUNS = False  # Italian is pro-drop; set True for an overt-pronoun comparison.
ADVERB_SELECTION = ('forse', 'probabilmente', 'certamente',
                    'apparentemente', 'fortunatamente')
MAX_TIME_SPECS = None               # Set to None for all time expressions.
MAX_ADVERBS = None                  # Set to None for all selected adverbs.

SUBJECT_NUMBER = 'both'          # 'sg', 'pl', or 'both'
SUBJECT_DET_TYPES = ('def',)     # 'def', 'indef', 'some', 'many', 'several', 'poss_my', ...
NOMINAL_TENSES = ('present_perfect',)
USE_ADVERBS_NOMINAL = True
RANDOM_SUBJECT_SELECTION = True
RANDOM_OUTPUT_SAMPLE = True
N_RANDOM_ROWS = 10
RANDOM_STATE = 99
REMOVE_DUPLICATE_SENTENCES = True

# Do not create passive sentences from non-passivizable expressions.
PASSIVE_EXCLUDED_VERBS = {'avere'}
PASSIVE_EXCLUDED_EXPRESSIONS = {'fare carriera'}

SOURCE_FIELDS = ('LF', 'keyverb_eng', 'verb_en', 'verb_it', 'object_it',
                 'gen_object_it', 'num_obj_it', 'collocation', 'full_expression')
COMMON_FIELDS = ('sentence', 'voice', 'tense', 'adverb', 'time_spec', 'subject',
                 'subject_realized', 'subject_type', 'person_key',
                 'subject_number', 'subject_gender', 'subject_det_type')
OUTPUT_FIELDS = SOURCE_FIELDS + COMMON_FIELDS


def read_csv(path):
    with open(path, encoding='utf-8-sig', newline='') as handle:
        reader = csv.DictReader(handle)
        assert reader.fieldnames, f'No column names in {path}'
        names = [name.strip().lstrip('\ufeff') for name in reader.fieldnames]
        if len(names) != len(set(names)):
            raise ValueError(f'Duplicate column names in {path}: {names}')
        return [{names[i]: (value or '').strip()
                 for i, value in enumerate(row.values())}
                for row in reader]


def unique_nonempty(values):
    return list(dict.fromkeys(str(value).strip() for value in values if str(value).strip()))


def load_inputs(input_dir=INPUT_DIR):
    input_dir = Path(input_dir)
    items = read_csv(input_dir / ITEMS_FILE)
    verbs = read_csv(input_dir / VERBS_FILE)
    builder = read_csv(input_dir / BUILDER_FILE)
    for field in SOURCE_FIELDS:
        if field not in (items[0] if items else {}):
            raise ValueError(f'Missing items column: {field}')
    for col in ('time_specs', 'adverbs', 'person_key', 'subjects_sg_def',
                'subjects_pl_def', 'person_sg', 'person_pl'):
        if col not in (builder[0] if builder else {}):
            raise ValueError(f'Missing builder column: {col}')
    for col in ('verb', 'person', *TENSE_COL.values(), *PASSIVE_TENSE_COL.values()):
        if col not in (verbs[0] if verbs else {}):
            raise ValueError(f'Missing verb-inflection column: {col}')
    index = {}
    for row in verbs:
        k = (row['verb'], row['person'])
        if k in index:
            raise ValueError(f'Duplicate verb-person inflection: {k}')
        index[k] = row
    return items, builder, index


def contexts(builder, max_time_specs=MAX_TIME_SPECS,
             max_adverbs=MAX_ADVERBS, adverb_selection=ADVERB_SELECTION):
    times = unique_nonempty(r['time_specs'] for r in builder)
    adverbs_available = unique_nonempty(r['adverbs'] for r in builder)
    if adverb_selection is None:
        adverbs = adverbs_available
    else:
        missing = set(adverb_selection) - set(adverbs_available)
        if missing:
            raise ValueError(f'Adverbs not in builder: {sorted(missing)}')
        adverbs = list(dict.fromkeys(adverb_selection))
    if max_time_specs is not None:
        times = times[:max_time_specs]
    if max_adverbs is not None:
        adverbs = adverbs[:max_adverbs]
    if not times:
        raise ValueError('No temporal expressions available')
    return times, adverbs


def expression_object(item):
    """Retain the original article/preposition in full_expression, fixing only whitespace.

    Return (object_text, error_message). Broken source templates are excluded
    and listed in the generator's audit CSV rather than guessed/repaired.
    """
    verb = item['verb_it']
    expression = re.sub(r'\s+', ' ', item['full_expression']).strip()
    if not verb or not expression:
        return '', 'missing verb or full_expression'
    if not expression.startswith(verb + ' '):
        return '', 'full_expression does not start with the source verb'
    obj = expression[len(verb):].strip()
    if not obj:
        return '', 'empty object phrase'
    if re.search(r'\b(?:da|di|a|con|per|su|in|tra|fra)$', obj, flags=re.I):
        return '', 'unfinished trailing preposition in full_expression'
    if re.search(r'\b(?:ART|MOD|N-I|N-II|N-III)\b', obj):
        return '', 'unresolved lexical-function placeholder'
    return obj, ''


def inflected(index, verb, key, tense, voice='active'):
    column = TENSE_COL[tense] if voice == 'active' else PASSIVE_TENSE_COL[tense]
    if (verb, key) not in index:
        raise ValueError(f'Missing inflection ({verb}, {key}, {column})')
    text = index[(verb, key)].get(column, '').strip()
    if not text:
        raise ValueError(f'Missing inflection ({verb}, {key}, {column})')
    return text


def make_active_sentence(time, subject, form, obj, tense, adverb=''):
    """Place the optional adverb after the time phrase, before the subject/verb.

    e.g. A gennaio, forse hanno fatto una scoperta.
         A gennaio, forse i ricercatori hanno fatto una scoperta.
    """
    parts = [time + ',', adverb, subject, form, obj]
    return ' '.join(part for part in parts if part) + '.'


def make_passive_sentence(time, form, obj, tense, adverb=''):
    """Place the optional adverb before the passive verb complex.

    e.g. A gennaio, forse è stata fatta una scoperta.
    """
    parts = [time + ',', adverb, form, obj]
    return ' '.join(part for part in parts if part) + '.'


def passive_key(item):
    gender = item['gen_object_it'].upper()
    number = item['num_obj_it'].lower()
    mapping = {('M', 'sg'): 'lui', ('F', 'sg'): 'lei',
               ('M', 'pl'): 'loro', ('F', 'pl'): 'loro_f'}
    if (gender, number) not in mapping:
        raise ValueError(f'Invalid object agreement: {gender}/{number}')
    return mapping[(gender, number)]


def passive_exclusion_reason(item, object_error=''):
    if object_error:
        return object_error
    if item['verb_it'] in PASSIVE_EXCLUDED_VERBS:
        return 'verb excluded from the personal passive (possessive avere)'
    expression = re.sub(r'\s+', ' ', item['full_expression']).strip()
    if expression in PASSIVE_EXCLUDED_EXPRESSIONS:
        return 'lexically non-passivizable or unnatural passive construction'
    return ''


def write_csv(path, rows, fieldnames=OUTPUT_FIELDS):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open('w', encoding='utf-8-sig', newline='') as handle:
        writer = csv.DictWriter(handle, fieldnames=fieldnames, extrasaction='ignore')
        writer.writeheader()
        writer.writerows(rows)
    return path


def output_row(item, sentence, voice, tense, adverb, time_spec,
               subject, person_key, subject_type, subject_number,
               subject_gender, subject_det_type='NA'):
    return {**{f: item.get(f, '') for f in SOURCE_FIELDS},
            'sentence': sentence, 'voice': voice, 'tense': tense,
            'adverb': adverb or 'NA', 'time_spec': time_spec, 'subject': subject,
            'subject_realized': 'yes' if subject else 'no',
            'subject_type': subject_type, 'person_key': person_key,
            'subject_number': subject_number, 'subject_gender': subject_gender,
            'subject_det_type': subject_det_type}


def finalize(rows, dedup=REMOVE_DUPLICATE_SENTENCES):
    if not dedup:
        return rows
    seen = set()
    out = []
    for row in rows:
        # Keep one stimulus per unique sentence and condition; original input files unchanged.
        key = (row['sentence'], row['collocation'], row['voice'], row['tense'])
        if key not in seen:
            seen.add(key)
            out.append(row)
    return out


def generate_active_pronominal(items, builder, index, output_dir=OUTPUT_DIR,
                                pronouns=ACTIVE_PRONOUN_KEYS,
                                realize_subject_pronouns=REALIZE_SUBJECT_PRONOUNS,
                                tenses=TENSES, max_time_specs=MAX_TIME_SPECS,
                                max_adverbs=MAX_ADVERBS,
                                adverb_selection=ADVERB_SELECTION):
    times, adverbs = contexts(builder, max_time_specs, max_adverbs, adverb_selection)
    pronoun_index = {r['person_key']: r for r in builder if r['person_key']}
    if not pronouns:
        raise ValueError('Set at least one pronoun person key')
    for p in pronouns:
        if p not in pronoun_index:
            raise ValueError(f'Person key {p} absent from builder')
    outputs = {}
    for tense in tenses:
        no_adv, with_adv = [], []
        for item in items:
            obj, error = expression_object(item)
            if error:
                continue
            for p in pronouns:
                meta = pronoun_index[p]
                subject = meta['subjects_pron'] if realize_subject_pronouns else ''
                subject_type = 'pronoun' if realize_subject_pronouns else 'null_pronoun'
                # Finite verb inflection still encodes person/number. Gender is
                # not recoverable from an omitted Italian subject pronoun.
                subject_gender = meta['pron_gender'] if realize_subject_pronouns else 'NA'
                form = inflected(index, item['verb_it'], p, tense)
                for time in times:
                    no_adv.append(output_row(item,
                        make_active_sentence(time, subject, form, obj, tense),
                        'active', tense, '', time, subject, p,
                        subject_type, meta['pron_number'], subject_gender))
                    for adv in adverbs:
                        with_adv.append(output_row(item,
                            make_active_sentence(time, subject, form, obj, tense, adv),
                            'active', tense, adv, time, subject, p,
                            subject_type, meta['pron_number'], subject_gender))
        for suffix, rows in [('no_adverbs', no_adv), ('with_adverbs', with_adv)]:
            path = Path(output_dir) / f'LVC_IT_active_{tense}_{suffix}.csv'
            out = finalize(rows)
            write_csv(path, out)
            outputs[path.name] = len(out)
    return outputs


def generate_passive(items, builder, index, output_dir=OUTPUT_DIR,
                     tenses=TENSES, max_time_specs=MAX_TIME_SPECS,
                     max_adverbs=MAX_ADVERBS,
                     adverb_selection=ADVERB_SELECTION):
    times, adverbs = contexts(builder, max_time_specs, max_adverbs, adverb_selection)
    outputs = {}
    for tense in tenses:
        no_adv, with_adv = [], []
        for item in items:
            obj, error = expression_object(item)
            if passive_exclusion_reason(item, error):
                continue
            p = passive_key(item)
            form = inflected(index, item['verb_it'], p, tense, voice='passive')
            for time in times:
                no_adv.append(output_row(item,
                    make_passive_sentence(time, form, obj, tense),
                    'passive', tense, '', time, obj, p, 'patient',
                    item['num_obj_it'], item['gen_object_it']))
                for adv in adverbs:
                    with_adv.append(output_row(item,
                        make_passive_sentence(time, form, obj, tense, adv),
                        'passive', tense, adv, time, obj, p, 'patient',
                        item['num_obj_it'], item['gen_object_it']))
        for suffix, rows in [('no_adverbs', no_adv), ('with_adverbs', with_adv)]:
            path = Path(output_dir) / f'LVC_IT_passive_{tense}_{suffix}.csv'
            out = finalize(rows)
            write_csv(path, out)
            outputs[path.name] = len(out)
    return outputs


def nominal_subjects(builder, numbers=SUBJECT_NUMBER, det_types=SUBJECT_DET_TYPES):
    """Use prepared Italian noun phrases; do not assemble a determiner from a list."""
    numbers = ('sg', 'pl') if numbers == 'both' else (numbers,)
    if set(numbers) - {'sg', 'pl'}:
        raise ValueError(f'Invalid SUBJECT_NUMBER {numbers}')
    options = []
    seen = set()
    for row in builder:
        if not row['subject_sg']:
            continue
        for number in numbers:
            gender = row.get(f'context_gender_{number}', row.get('gender_sbj', 'M'))
            p = row.get('person_' + number, '')
            for det_type in det_types:
                if det_type == 'def':
                    field = f'subjects_{number}_def'
                elif det_type == 'indef' and number == 'sg':
                    field = 'subjects_sg_indef'
                elif det_type in ('some', 'many', 'several') and number == 'pl':
                    field = f'subjects_pl_{det_type}'
                elif det_type.startswith('poss_'):
                    poss = det_type[5:]
                    if poss not in ('my', 'your', 'his', 'her', 'our', 'their'):
                        raise ValueError(f'Unknown possessive option {det_type}')
                    field = f'subjects_{number}_poss_{poss}'
                else:
                    continue
                subject = row.get(field, '').strip()
                if not subject:
                    continue
                key = (subject, number, p, det_type)
                if key in seen:
                    continue
                seen.add(key)
                options.append({'subject': subject, 'person_key': p,
                                'number': number, 'gender': gender,
                                'subject_det_type': det_type})
    if not options:
        raise ValueError('No nominal subjects: check SUBJECT_NUMBER and SUBJECT_DET_TYPES')
    return options


def generate_active_nominal(items, builder, index, output_dir=OUTPUT_DIR,
                            numbers=SUBJECT_NUMBER, det_types=SUBJECT_DET_TYPES,
                            tenses=NOMINAL_TENSES,
                            use_adverbs=USE_ADVERBS_NOMINAL,
                            random_subject_selection=RANDOM_SUBJECT_SELECTION,
                            random_output_sample=RANDOM_OUTPUT_SAMPLE,
                            n_random_rows=N_RANDOM_ROWS,
                            random_state=RANDOM_STATE,
                            max_time_specs=MAX_TIME_SPECS,
                            max_adverbs=MAX_ADVERBS,
                            adverb_selection=ADVERB_SELECTION):
    rng = random.Random(random_state)
    subjects = nominal_subjects(builder, numbers, det_types)
    times, adverbs = contexts(builder, max_time_specs, max_adverbs, adverb_selection)
    rows = []
    for item in items:
        obj, error = expression_object(item)
        if error:
            continue
        for tense in tenses:
            for time in times:
                chosen = [rng.choice(subjects)] if random_subject_selection else subjects
                for subj in chosen:
                    form = inflected(index, item['verb_it'], subj['person_key'], tense)
                    modifiers = [''] + adverbs if use_adverbs else ['']
                    for adv in modifiers:
                        rows.append(output_row(item,
                            make_active_sentence(time, subj['subject'], form, obj, tense, adv),
                            'active', tense, adv, time, subj['subject'], subj['person_key'],
                            'nominal', subj['number'], subj['gender'], subj['subject_det_type']))
    out = finalize(rows)
    if random_output_sample:
        out = rng.sample(out, k=min(n_random_rows, len(out)))
    path = Path(output_dir) / 'LVC_IT_active_nominal_subjects.csv'
    write_csv(path, out)
    return {path.name: len(out)}


def write_audit(items, output_dir=OUTPUT_DIR):
    rows = []
    for idx, item in enumerate(items, start=2):
        obj, err = expression_object(item)
        if err:
            rows.append({'source_csv_row': idx, **{k: item.get(k, '') for k in SOURCE_FIELDS},
                         'scope': 'all_generators', 'reason': err})
        elif passive_exclusion_reason(item):
            rows.append({'source_csv_row': idx, **{k: item.get(k, '') for k in SOURCE_FIELDS},
                         'scope': 'passive_only',
                         'reason': passive_exclusion_reason(item)})
    filename = Path(output_dir) / 'LVC_IT_generation_exclusions.csv'
    write_csv(filename, rows, ('source_csv_row', *SOURCE_FIELDS, 'scope', 'reason'))
    return filename, Counter(r['reason'] for r in rows)


def generate_all(items, builder, index, output_dir=OUTPUT_DIR):
    output_dir = Path(output_dir)
    a = generate_active_pronominal(items, builder, index, output_dir=output_dir)
    p = generate_passive(items, builder, index, output_dir=output_dir)
    n = generate_active_nominal(items, builder, index, output_dir=output_dir)
    audit, reasons = write_audit(items, output_dir)
    return a, p, n, audit, reasons



items, builder, verb_index = load_inputs()
print(f'Loaded {len(items)} items and {len(builder)} builder rows.')


## 1. Active sentences — null subject by default

Italian normally omits the subject pronoun in neutral contexts. With `REALIZE_SUBJECT_PRONOUNS = False`, the output looks like *A gennaio hanno fatto una scoperta.* The CSV records `subject = ""`, `subject_realized = no`, `subject_type = null_pronoun`, `person_key = loro`, `subject_number = pl` and `subject_gender = NA`. This retains morphological information without asserting the gender of a non-overt subject. For a matched overt condition, set `REALIZE_SUBJECT_PRONOUNS = True` and specify another output folder.


In [ ]:
active_counts = generate_active_pronominal(items, builder, verb_index, output_dir=OUTPUT_DIR)
for filename, count in active_counts.items():
    print(f'{filename}: {count:,} sentences')


## 2. Passive sentences

Passive participles agree with the **object** in gender/number. The object becomes the patient subject and normally follows the verbal complex (e.g. *A gennaio è stata fatta una scoperta*). *Avere* and *fare carriera* are excluded. This section also saves the exclusions audit.


In [ ]:
passive_counts = generate_passive(items, builder, verb_index, output_dir=OUTPUT_DIR)
audit_path, audit_summary = write_audit(items, OUTPUT_DIR)
for filename, count in passive_counts.items():
    print(f'{filename}: {count:,} sentences')
print('Audit:', audit_path, dict(audit_summary))


## 3. Active sentences — nominal subjects

Default settings mirror the French workflow: random selection of one nominal subject per item × temporal phrase × tense, and a reproducible **10-row sample**. Set `RANDOM_OUTPUT_SAMPLE = False` for the full nominal output. Options for `SUBJECT_DET_TYPES` include `def`, `indef` (sg only), `some`, `many`, `several` (pl only), and `poss_my`, `poss_your`, `poss_his`, `poss_her`, `poss_our`, `poss_their` (only where builder permits).


In [ ]:
nominal_counts = generate_active_nominal(items, builder, verb_index, output_dir=OUTPUT_DIR)
for filename, count in nominal_counts.items():
    print(f'{filename}: {count:,} sentences')


## 4. Basic grammatical checks and examples

The optional adverb is placed **after the temporal phrase and before the finite verb** (and before an overt nominal or pronominal subject, if present). This placement is standardized across active and passive conditions.


In [ ]:
assert expression_object({'verb_it':'fare', 'full_expression':'fare un errore'})[0] == 'un errore'
assert expression_object({'verb_it':'ricevere', 'full_expression':'ricevere un abbraccio da'})[1]
assert REALIZE_SUBJECT_PRONOUNS is False
assert inflected(verb_index, 'fare', 'loro', 'present_perfect') == 'hanno fatto'
assert make_active_sentence('A gennaio', '', 'hanno fatto', 'una scoperta', 'present_perfect') == 'A gennaio, hanno fatto una scoperta.'
assert make_active_sentence('A gennaio', '', 'hanno fatto', 'una scoperta', 'present_perfect', 'forse') == 'A gennaio, forse hanno fatto una scoperta.'
assert make_active_sentence('A gennaio', '', 'faranno', 'una scoperta', 'future') == 'A gennaio, faranno una scoperta.'
assert make_active_sentence('A gennaio', 'loro', 'hanno fatto', 'una scoperta', 'present_perfect', 'forse') == 'A gennaio, forse loro hanno fatto una scoperta.'
assert make_active_sentence('A gennaio', 'loro', 'hanno fatto', 'una scoperta', 'present_perfect') == 'A gennaio, loro hanno fatto una scoperta.'
assert inflected(verb_index, 'fare', 'lei', 'present_perfect', 'passive') == 'è stata fatta'
assert make_passive_sentence('A gennaio', 'è stata fatta', 'una scoperta', 'present_perfect') == 'A gennaio, è stata fatta una scoperta.'
assert make_passive_sentence('A gennaio', 'sarà presa', 'una decisione', 'future', 'forse') == 'A gennaio, forse sarà presa una decisione.'
assert make_passive_sentence('A gennaio', 'è stata presa', 'una decisione', 'present_perfect', 'forse') == 'A gennaio, forse è stata presa una decisione.'
assert passive_key({'gen_object_it':'F','num_obj_it':'pl'}) == 'loro_f'
assert inflected(verb_index, 'prendere', 'loro_f', 'present_perfect', 'passive') == 'sono state prese'
assert make_passive_sentence('A gennaio', 'sono state prese', 'le misure', 'present_perfect') == 'A gennaio, sono state prese le misure.'
rows = read_csv(OUTPUT_DIR / 'LVC_IT_active_present_perfect_no_adverbs.csv')
assert rows and all(r['subject'] == '' and r['subject_realized'] == 'no' and r['person_key'] == 'loro' and r['subject_number'] == 'pl' for r in rows)
print('All basic checks passed.')
for name in ['LVC_IT_active_present_perfect_no_adverbs.csv',
             'LVC_IT_passive_present_perfect_no_adverbs.csv',
             'LVC_IT_active_nominal_subjects.csv']:
    samples = read_csv(OUTPUT_DIR / name)
    print(name)
    for row in samples[:3]: print(' •', row['sentence'])


- The default Italian active output **omits the subject pronoun**, unlike French or German; this improves neutrality but introduces a cross-linguistic surface difference.
- Person and number are **retained** via `person_key` and `subject_number`, even with `subject_realized = no`. Gender is marked `NA` for null active subjects.
- Set `REALIZE_SUBJECT_PRONOUNS = True` to generate a directly matched overt-pronoun comparison; use a **different output folder** to avoid overwriting the main condition.
- All sentence conditions place adverbs before the verbal complex (and before overt subjects, when present). Other generation settings are unchanged.
- Review generated passives for lexical naturalness, and time specifications for semantic compatibility.
- Duplicate input items are deduplicated by sentence/condition; original inputs remain unchanged.
- `ricevere un abbraccio da` remains excluded until its source phrase is completed.
- Nominal-subject translations were proposed automatically and require review.
